### 3 Approaches to Tokenisation 

#### Approach 1: Word-level Tokenisation 
* Split on spaces and punctuations
* Cons: Requires massive vocabulary to cover every possible word & miss a word, it is tokenised as [UNK]

#### Approach 2: Character-level Tokenisation 
* Split words at every character
* Pros: Vocab is very small 
* Cons: Sequences are extremely long, 10 words becomes ~ 50 character level tokens
* Cons 2: Model needs to learn that 't' 'h' 'e' means 'the' 

#### Approach 3: Subword Tokenisation
* Sweet spot to decompose words into menaningful and re-usable pieces 
* unhappiness = 'un', 'happi', 'ness' 
* Pros: Vocab stays managable as compared to word-level tokenisation & minimal unknown tokens as most words can be built from the subwords present in the dictionary 

### Byte-pair Encoding
Start with individual characters. Count every adjacent pair in the training corpus. Merge the most frequent pair into a new token. Repeat until you reach the target vocabulary size.

### Implementation of BPE

In [1]:
from collections import Counter

In [5]:
class BPETokenizer:
    def __init__(self):
        self.merges = {}
        self.vocab = {}
    
    def _get_pairs(self, tokens):
        pairs = Counter()
        for i in range(len(tokens) - 1):
            pairs[(tokens[i], tokens[i + 1])] += 1
        return pairs
    
    def _merge_pair(self, tokens, pair, new_token):
        merged = []
        i = 0
        while i < len(tokens):
            if i < len(tokens) - 1 and tokens[i] == pair[0] and tokens[i + 1] == pair[1]:
                merged.append(new_token)
                i += 2
            else:
                merged.append(tokens[i])
                i += 1
        return merged
    
    def train(self, text, num_merges):
        tokens = list(text.encode("utf-8"))
        self.vocab = {i: bytes([i]) for i in range(256)}

        for i in range(num_merges):
            pairs = self._get_pairs(tokens)
            if not pairs:
                break
            best_pair = max(pairs, key = pairs.get)
            new_token = 256 + i
            tokens = self._merge_pair(tokens, best_pair, new_token)
            self.merges[best_pair] = new_token
            self.vocab[new_token] = self.vocab[best_pair[0]] + self.vocab[best_pair[1]]
        
        return self
    
    def encode(self, text):
        tokens = list(text.encode('utf-8'))
        for pair, new_token in self.merges.items():
            tokens = self._merge_pair(tokens, pair, new_token)
        return tokens
    
    def decode(self, tokens):
        byte_sequence = b"".join(self.vocab[t] for t in tokens)
        return byte_sequence.decode('utf-8', errors='replace')

In [6]:
corpus = (
    "The cat sat on the mat. The cat ate the rat. "
    "The dog sat on the log. The dog ate the frog. "
    "Natural language processing is the study of how computers "
    "understand and generate human language. "
    "Tokenization is the first step in any NLP pipeline."
)

tokenizer = BPETokenizer()
tokenizer.train(corpus, num_merges=40)

test_sentences = [
    "The cat sat on the mat.",
    "Natural language processing",
    "tokenization pipeline",
    "unhappiness",
]

for sentence in test_sentences:
    encoded = tokenizer.encode(sentence)
    decoded = tokenizer.decode(encoded)
    raw_bytes = len(sentence.encode("utf-8"))
    ratio = len(encoded) / raw_bytes
    print(f"'{sentence}'")
    print(f"  Tokens: {len(encoded)} (from {raw_bytes} bytes) -- ratio: {ratio:.2f}")
    print(f"  Roundtrip: {'PASS' if decoded == sentence else 'FAIL'}")

'The cat sat on the mat.'
  Tokens: 3 (from 23 bytes) -- ratio: 0.13
  Roundtrip: PASS
'Natural language processing'
  Tokens: 17 (from 27 bytes) -- ratio: 0.63
  Roundtrip: PASS
'tokenization pipeline'
  Tokens: 16 (from 21 bytes) -- ratio: 0.76
  Roundtrip: PASS
'unhappiness'
  Tokens: 10 (from 11 bytes) -- ratio: 0.91
  Roundtrip: PASS


In [16]:
tokenizer.encode("Nothing")

[78, 111, 116, 104, 286, 103]

In [ ]:
""